# 1. Carte thématique du Togo

Une valeur par **région**, **préfecture** ou **canton**, pour un indicateur et une saison :

| Mesure | Ce qui est affiché |
|---|---|
| Moyenne de référence | moyenne de la saison sur la période de référence (2001-2020) |
| Dernière année complète | valeur de la dernière année où ≥ 90 % des unités ont une donnée |
| Écart à la référence | dernière année complète − moyenne de référence |
| Tendance | pente de Theil-Sen par décennie ; **grisée si non significative** (Mann-Kendall, α = 0,05) |

Les unités sans valeur sont en gris. Les 7 sites pilotes sont les points noirs.
Source : `data/api/` (étape `export` du pipeline). Rien n'est recalculé ici.

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as w
from IPython.display import JSON, Markdown, display

import agbea_nb as nb

pd.set_option("display.max_rows", 60)
cat = nb.catalog()
print(f"Données : {nb.API}")
print(f"Générées le {cat['generated_at']} · pipeline {cat['pipeline_version']} · référence {cat['baseline']}")

In [ ]:
INDICATEURS = ["rainfall", "ndvi", "lst_day", "lst_night", "elevation"]
MESURES = {"Moyenne de référence": "baseline", "Dernière année complète": "last",
           "Écart à la référence (dernière année)": "anomaly", "Tendance (pente de Sen / décennie)": "trend"}


def valeurs(level, ind, saison, mesure):
    """Tableau unit_id, name, value (+ colonnes d'information) et titre de la carte."""
    u = nb.units().query("level == @level")[["unit_id", "name"]]
    unit = cat["indicators"][ind]["unit"]
    if ind == "elevation":
        e = nb.table("elevation")[["unit_id", "mean", "min", "max", "pct_below_10m"]]
        return u.merge(e.rename(columns={"mean": "value"}), on="unit_id", how="left"), "Altitude moyenne (NASADEM), m", unit
    if mesure == "trend":
        t = nb.table("trends").query("indicator == @ind and season == @saison")
        t = t[["unit_id", "sen_slope_per_decade", "p_value", "significant", "first_year", "last_year"]]
        d = u.merge(t, on="unit_id", how="left").assign(value=lambda x: x.sen_slope_per_decade)
        per = (f"{int(t.first_year.min())}-{int(t.last_year.max())}" if t.first_year.notna().any() else "période inconnue")
        return d, f"{nb.label(ind)} — tendance par décennie, {nb.SEASON_FR[saison]}, {per}", f"{unit} / décennie"
    s = nb.table("seasonal").query("indicator == @ind and season == @saison")
    if mesure == "baseline":
        d = s.groupby("unit_id", as_index=False).baseline_mean.first().rename(columns={"baseline_mean": "value"})
        titre = f"{nb.label(ind)} — moyenne {cat['baseline']}, {nb.SEASON_FR[saison]}"
    else:
        year = nb.last_complete_year(ind, saison, level)
        if year is None:
            return u.assign(value=np.nan), f"{nb.label(ind)} — aucune année complète ({nb.SEASON_FR[saison]})", unit
        col = "value" if mesure == "last" else "anomaly"
        d = s[s.year == year][["unit_id", col, "n_months"]].rename(columns={col: "value"})
        titre = f"{nb.label(ind)} — {'valeur' if mesure == 'last' else 'écart à la référence'} {year}, {nb.SEASON_FR[saison]}"
    return u.merge(d, on="unit_id", how="left"), titre, unit


def carte(level="region", ind="rainfall", saison="annual", mesure="Moyenne de référence", sites=True):
    code = MESURES[mesure]
    d, titre, unit = valeurs(level, ind, saison, code)
    gj = nb.geojson(level)
    seq, div = nb.SCALES.get(ind, ("Viridis", "RdBu_r"))
    diverging = code in ("anomaly", "trend") and ind != "elevation"
    colored = d.dropna(subset=["value"])
    if code == "trend" and ind != "elevation":
        colored = colored[colored.significant.fillna(False).astype(bool)]
    grey = d[~d.unit_id.isin(colored.unit_id)]

    fig = go.Figure()
    if len(colored):
        fig.add_trace(go.Choroplethmap(
            geojson=gj, featureidkey="properties.unit_id", locations=colored.unit_id, z=colored.value,
            colorscale=div if diverging else seq, zmid=0 if diverging else None,
            marker_opacity=0.8, marker_line_width=0.6, text=colored.name,
            hovertemplate="<b>%{text}</b><br>%{z:.3g} " + unit + "<extra></extra>",
            colorbar=dict(title=unit), name="valeur"))
    if len(grey):
        why = "pas de tendance significative" if code == "trend" else "pas de valeur"
        fig.add_trace(go.Choroplethmap(
            geojson=gj, featureidkey="properties.unit_id", locations=grey.unit_id, z=np.zeros(len(grey)),
            colorscale=[[0, "#cfcfcf"], [1, "#cfcfcf"]], showscale=False, marker_opacity=0.55,
            marker_line_width=0.6, text=grey.name, hovertemplate="<b>%{text}</b><br>" + why + "<extra></extra>",
            name=why))
    if sites:
        s = nb.units().query("level == 'site'")
        fig.add_trace(go.Scattermap(lat=s.centroid_lat, lon=s.centroid_lon, mode="markers+text", text=s.name,
                                    textposition="top right", marker=dict(size=9, color="black"),
                                    name="Sites pilotes", hoverinfo="text"))
    fig.update_layout(title=titre, height=760, margin=dict(l=0, r=0, t=50, b=0), showlegend=False,
                      map=dict(style="carto-positron", center=nb.TOGO_CENTER, zoom=5.7))
    fig.show()
    cols = [c for c in d.columns if c != "unit_id"]
    display(d.sort_values("value", ascending=False)[["unit_id", *cols]].round(3).reset_index(drop=True))

## Carte interactive
Changer les menus redessine la carte et le tableau trié en dessous.

In [ ]:
w.interact(
    carte,
    level=w.Dropdown(options=[("Région", "region"), ("Préfecture", "prefecture"), ("Canton", "canton")], value="region", description="Niveau"),
    ind=w.Dropdown(options=[(nb.label(i), i) for i in INDICATEURS], value="rainfall", description="Indicateur"),
    saison=w.Dropdown(options=[(v, k) for k, v in nb.SEASON_FR.items()], value="annual", description="Saison"),
    mesure=w.Dropdown(options=list(MESURES), value="Moyenne de référence", description="Mesure"),
    sites=w.Checkbox(value=True, description="Sites pilotes"),
);